# Latent feature extraction - one common latent space for the four diseases and the cross-hospital ICU cohort

**Input:** the final preprocessed train / validation / test CSVs of `heart`, `diabetes`, `hypertension` and `respiratory`
(`<dataset>/preprocessed/`) and of the cross-hospital ICU cohort `hospital` (`hospital/preprocessed/`: eICU-CRD v2.0 split by
whole hospitals, plus MIMIC-IV v3.1 as an untouched `external` hospital; four recorded-condition labels `target_heart`,
`target_diabetes`, `target_hypertension`, `target_respiratory`). **Output:** `latent/data/combined_latent_<split>.csv` (all five
datasets in one file, column `dataset`), `latent/data/<dataset>_latent_<split>.csv` (plus `hospital_latent_external.csv` for
MIMIC), and one dashboard per metric in `latent/plots/`.

**Methods (latent methods.png):** DGCCA, SAINT and MV-DAE learn one shared 16-dimensional latent space (each dataset has its own
input encoder because the predictors differ; the hospital view has a 4-output label head, one per recorded condition). A
per-dataset PCA without any shared space is the **before** reference. MIMIC is never used for training, checkpoints or metrics;
it is only encoded so that the in-processing cross-hospital bias can be checked on a different hospital system.

**Metrics (latent metrics.png) - they decide which method is best:**

| Metric | Measures | Better |
|---|---|---|
| Cross-dataset alignment (CKA) | how well the datasets' latent features align in the common space (centroids of shared age x sex x BMI strata; age x sex for pairs with the hospital cohort, which has no BMI; linear CKA averaged over dataset pairs) | higher |
| Clinical information preservation (Macro-F1) | class-balanced logistic probe on the latent features, per dataset (hospital: mean over its four conditions) | higher |
| Latent structure (Silhouette) | separation of the clinical classes in the latent space | higher |
| Domain invariance (NMI, ARI) | k-means clusters of the pooled latents vs dataset identity | lower |

The selected method has the best mean rank over the five metrics (validation rows; test rows are only encoded).

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML
LATENT = Path(r"D:/complete/latent")
sys.path.insert(0, str(LATENT))
import latent_alignment as la
RUN = True                      # trains DGCCA, SAINT and MV-DAE (several minutes)
if RUN:
    la.main()

## Method comparison (every method x every metric; lower mean rank = better)

In [ ]:
report = json.loads((LATENT / "latent_report.json").read_text())
print("Selected latent method:", report["selected"], "|", report["rule"])
table = pd.read_csv(LATENT / "latent_method_comparison.csv", index_col=0)
display(table.style.format(precision=4).apply(lambda r: ["background-color:#e6f6e7;font-weight:bold" if r.name == report["selected"] else "" for _ in r], axis=1))

## Per-dataset Macro-F1 / AUROC / Silhouette (validation)

In [ ]:
rows = []
for m, r in report["methods"].items():
    for d, v in r["per_dataset"].items():
        rows.append({"method": m, "dataset": d, **v, "uncompressed macro_f1": report["uncompressed_macro_f1"][d]})
display(pd.DataFrame(rows).set_index(["method", "dataset"]).round(4))

## Pairwise cross-dataset CKA of the selected method

In [ ]:
sel = report["methods"][report["selected"]]
display(pd.DataFrame(sel["cka_matrix"], index=la.DATASETS, columns=la.DATASETS).round(3))

## Dashboards and output files

In [ ]:
display(HTML("<br>".join(f'<a href="{p.as_uri()}" target="_blank">{p.name}</a>' for p in sorted((LATENT / "plots").glob("*.html")))))
display(pd.read_csv(LATENT / "data" / "combined_latent_train.csv", nrows=5))
print({p.name: f"{p.stat().st_size/1e6:.1f} MB" for p in sorted((LATENT / "data").glob("*.csv"))})